In [22]:
import pandas as pd 

MainAtApogee_df = pd.read_csv('/Users/ricechessmaster/Documents/Dispersion Analysis v2/backend/data/Polaris_Cycle1_Aug2025-MainAtApogee.csv')
MainAtApogee_df = MainAtApogee_df.set_index('Simulation')

DrogueOnly_df = pd.read_csv('/Users/ricechessmaster/Documents/Dispersion Analysis v2/backend/data/Polaris_Cycle1_Aug2025-DrogueOnly.csv')
DrogueOnly_df = DrogueOnly_df.set_index('Simulation')

Ballistic_df = pd.read_csv('/Users/ricechessmaster/Documents/Dispersion Analysis v2/backend/data/Polaris_Cycle1_Aug2025-Ballistic.csv')
Ballistic_df = Ballistic_df.set_index('Simulation')

In [23]:
SimParams_df = pd.read_csv('/Users/ricechessmaster/Documents/Dispersion Analysis v2/backend/data/sim_parameters_historical_aug2025_filtered.csv')
altitudes = [110, 320, 500, 800, 1000, 1500, 1900, 3200, 4200, 5600, 7200, 9200, 10400, 11800, 13500, 15800, 17700, 19300, 22000]
SimParams_df = SimParams_df.rename(columns={f"direction [{alt}]": f"direction_{alt}" for alt in altitudes})
SimParams_df = SimParams_df.drop(columns=[f"stdev [{alt}]" for alt in altitudes])


In [24]:
# Order of when to join would matter eventually if there is too much data; could apply where first, then join
# left_index = True is used as the name "Simulations" for the 0th column is not real... it's a default

Full_MainAtApogee_df = pd.merge(MainAtApogee_df, SimParams_df, left_index=True, right_on='date', how="inner")
Full_DrogueOnly_df = pd.merge(DrogueOnly_df, SimParams_df, left_index=True, right_on='date', how="inner")
Full_Ballistic_df = pd.merge(Ballistic_df, SimParams_df, left_index=True, right_on='date', how="inner")


In [25]:
import numpy as np


def haversine_nm(ref_latitude:float, ref_longitude:float, latitude, longitude):
    """
    Compute the great-circle distance between two points on Earth (in nautical miles).
    """

    # Earth's mean radius in nautical miles
    R_nm = 3443.92

    # Note: input of 2 scalars and 2 arrays to be converted to radians
    ref_latitude, ref_longitude, latitude, longitude = map(np.radians, [ref_latitude, ref_longitude, latitude, longitude])

    dlat = latitude - ref_latitude
    dlon = longitude - ref_longitude

    a = np.sin(dlat / 2) ** 2 + np.cos(ref_latitude) * np.cos(latitude) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return R_nm * c


# reference launch coordinates (degrees)
ref_lat = 47.965378
ref_long = -81.873536

Main_landing_latitudes = Full_MainAtApogee_df["Polaris Rocket Landing Latitude (°N)"].to_numpy()
Main_landing_longitudes = Full_MainAtApogee_df["Polaris Rocket Landing Longitude (°E)"].to_numpy()

# compute great-circle distance in nautical miles using the shared haversine helper
landing_distances = haversine_nm(ref_lat, ref_long, Main_landing_latitudes, Main_landing_longitudes)

Full_MainAtApogee_df['distance_nm'] = landing_distances
Main_safety_column = np.where(landing_distances < 10, True, False)
Full_MainAtApogee_df["safety_column"] = Main_safety_column
print(Full_MainAtApogee_df.columns.tolist())

['Max Windspeed (mph)', 'Wind Direction (deg)', 'Temperature (°C)', 'Pressure (mbar)', 'Apogee (ft)', 'Max Mach', 'Polaris Rocket Initial Stability', 'Polaris Rocket Min Stability', 'Polaris Rocket Max Stability', 'Polaris Rocket Apogee Stability', 'Polaris Rocket Landing Latitude (°N)', 'Polaris Rocket Landing Longitude (°E)', 'Polaris Rocket Position East of Launch (ft)', 'Polaris Rocket Position North of Launch (ft)', 'Polaris Rocket Lateral Velocity at Apogee (m/s)', 'date', 'temperature', 'pressure', '110', 'direction_110', '320', 'direction_320', '500', 'direction_500', '800', 'direction_800', '1000', 'direction_1000', '1500', 'direction_1500', '1900', 'direction_1900', '3200', 'direction_3200', '4200', 'direction_4200', '5600', 'direction_5600', '7200', 'direction_7200', '9200', 'direction_9200', '10400', 'direction_10400', '11800', 'direction_11800', '13500', 'direction_13500', '15800', 'direction_15800', '17700', 'direction_17700', '19300', 'direction_19300', '22000', 'directi

In [26]:
DrogueOnly_landing_latitudes = Full_DrogueOnly_df["Polaris Rocket Landing Latitude (°N)"].to_numpy()
DrogueOnly_landing_longitudes = Full_DrogueOnly_df["Polaris Rocket Landing Longitude (°E)"].to_numpy()

# compute great-circle distance in nautical miles using the shared haversine helper
landing_distances = haversine_nm(ref_lat, ref_long, DrogueOnly_landing_latitudes, DrogueOnly_landing_longitudes)

# insert distances and safety column into dataframe
Full_DrogueOnly_df['distance_nm'] = landing_distances
DrogueOnly_safety_column = np.where(landing_distances < 10, True, False)
Full_DrogueOnly_df["safety_column"] = DrogueOnly_safety_column

In [27]:
# Extract and use the haversine function to compute distance from the starting point
Ballistic_landing_latitudes = Full_Ballistic_df["Polaris Rocket Landing Latitude (°N)"].to_numpy()
Ballistic_landing_longitudes = Full_Ballistic_df["Polaris Rocket Landing Longitude (°E)"].to_numpy()

# compute great-circle distance in nautical miles using the shared haversine helper
landing_distances = haversine_nm(ref_lat, ref_long, Ballistic_landing_latitudes, Ballistic_landing_longitudes)

# insert distances and safety column into dataframe
Full_Ballistic_df['distance_nm'] = landing_distances
Ballistic_safety_column = np.where(landing_distances < 10, True, False)
Full_Ballistic_df["safety_column"] = Ballistic_safety_column

In [28]:
Full_safety_col_arry = Full_MainAtApogee_df["safety_column"].to_numpy()
DrogueOnly_safety_col_arry = Full_DrogueOnly_df["safety_column"].to_numpy()
Ballistic_safety_col_arry = Full_Ballistic_df["safety_column"].to_numpy()

# Default axis to None, which would flatten the array either way
safe, unsafe  = np.count_nonzero(Full_safety_col_arry == True), np.count_nonzero(Full_safety_col_arry == False)
print(safe, unsafe)

safe, unsafe  = np.count_nonzero(DrogueOnly_safety_col_arry == True), np.count_nonzero(DrogueOnly_safety_col_arry == False)
print(safe, unsafe)

safe, unsafe  = np.count_nonzero(Ballistic_safety_col_arry == True), np.count_nonzero(Ballistic_safety_col_arry == False)
print(safe, unsafe)


227 1
228 0
228 0


In [29]:
# Changing all df names to the db names 

dataframes = [Full_MainAtApogee_df, Full_DrogueOnly_df, Full_Ballistic_df]

for idx in range(len(dataframes)):
    table = dataframes[idx].rename(columns={
        'Temperature (°C)': 'temperature_sim',
        'Pressure (mbar)': 'pressure_sim',
        'Polaris Rocket Landing Latitude (°N)': 'landing_lat',
        'Polaris Rocket Landing Longitude (°E)': 'landing_lon'
    })

    for alt in altitudes:
          # this fills the column with the string "wind_speed_110"
        table = table.rename(columns={
            f"{alt}" : f"wind_speed_{alt}", 
            f"direction_{alt}" : f"wind_dir_{alt}"
         })
        
    dataframes[idx] = table
    
Full_MainAtApogee_df, Full_DrogueOnly_df, Full_Ballistic_df = dataframes
print(Full_Ballistic_df.columns.tolist())

['Max Windspeed (mph)', 'Wind Direction (deg)', 'temperature_sim', 'pressure_sim', 'Apogee (ft)', 'Max Mach', 'Polaris Rocket Initial Stability', 'Polaris Rocket Min Stability', 'Polaris Rocket Max Stability', 'Polaris Rocket Apogee Stability', 'landing_lat', 'landing_lon', 'Polaris Rocket Position East of Launch (ft)', 'Polaris Rocket Position North of Launch (ft)', 'Polaris Rocket Lateral Velocity at Apogee (m/s)', 'date', 'temperature', 'pressure', 'wind_speed_110', 'wind_dir_110', 'wind_speed_320', 'wind_dir_320', 'wind_speed_500', 'wind_dir_500', 'wind_speed_800', 'wind_dir_800', 'wind_speed_1000', 'wind_dir_1000', 'wind_speed_1500', 'wind_dir_1500', 'wind_speed_1900', 'wind_dir_1900', 'wind_speed_3200', 'wind_dir_3200', 'wind_speed_4200', 'wind_dir_4200', 'wind_speed_5600', 'wind_dir_5600', 'wind_speed_7200', 'wind_dir_7200', 'wind_speed_9200', 'wind_dir_9200', 'wind_speed_10400', 'wind_dir_10400', 'wind_speed_11800', 'wind_dir_11800', 'wind_speed_13500', 'wind_dir_13500', 'wind_

In [30]:
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

column_names = ["temperature", "pressure"]
altitudes = [110, 320, 500, 800, 1000, 1500, 1900, 3200, 4200, 5600, 7200, 9200, 10400, 11800, 13500, 15800, 17700, 19300, 22000]

for alt in altitudes:
    column_names.append(f"wind_speed_{alt}")
    column_names.append(f"wind_dir_{alt}")

# Define the input (the input data is always the same)
X_MainAtApogee = Full_MainAtApogee_df[column_names]
X_DrogueOnly = Full_DrogueOnly_df[column_names]
X_Ballistic = Full_Ballistic_df[column_names]

# Targets
y_Full_MainAtApogee_distance = Full_MainAtApogee_df['distance_nm']
y_Full_MainAtApogee_latitude = Full_MainAtApogee_df['landing_lat']
y_Full_MainAtApogee_longitude = Full_MainAtApogee_df['landing_lon']

y_Full_DrogueOnly_distance = Full_DrogueOnly_df['distance_nm']
y_Full_DrogueOnly_latitude = Full_DrogueOnly_df['landing_lat']
y_Full_DrogueOnly_longitude = Full_DrogueOnly_df['landing_lon']

y_Full_Ballistic_distance = Full_Ballistic_df['distance_nm']
y_Full_Ballistic_latitude = Full_Ballistic_df['landing_lat']
y_Full_Ballistic_longitude = Full_Ballistic_df['landing_lon']

In [31]:
# 80% for training, set seed 42; stratify not necessary
# MainAtApogee
X_train_main, X_test_main, y_train_main_distance, y_test_main_distance = train_test_split(X_MainAtApogee, y_Full_MainAtApogee_distance, test_size=0.20, random_state=42)
X_train_main, X_test_main, y_train_main_latitude, y_test_main_latitude = train_test_split(X_MainAtApogee, y_Full_MainAtApogee_latitude, test_size=0.20, random_state=42)
X_train_main, X_test_main, y_train_main_longitude, y_test_main_longitude = train_test_split(X_MainAtApogee, y_Full_MainAtApogee_longitude, test_size=0.20, random_state=42)

# DrogueOnly
X_train_drogue, X_test_drogue, y_train_drogue_distance, y_test_drogue_distance = train_test_split(X_DrogueOnly, y_Full_DrogueOnly_distance, test_size=0.20, random_state=42)
X_train_drogue, X_test_drogue, y_train_drogue_latitude, y_test_drogue_latitude = train_test_split(X_DrogueOnly, y_Full_DrogueOnly_latitude, test_size=0.20, random_state=42)
X_train_drogue, X_test_drogue, y_train_drogue_longitude, y_test_drogue_longitude = train_test_split(X_DrogueOnly, y_Full_DrogueOnly_longitude, test_size=0.20, random_state=42)

# Ballistic
X_train_ballistic, X_test_ballistic, y_train_ballistic_distance, y_test_ballistic_distance = train_test_split(X_Ballistic, y_Full_Ballistic_distance, test_size=0.20, random_state=42)
X_train_ballistic, X_test_ballistic, y_train_ballistic_latitude, y_test_ballistic_latitude = train_test_split(X_Ballistic, y_Full_Ballistic_latitude, test_size=0.20, random_state=42)
X_train_ballistic, X_test_ballistic, y_train_ballistic_longitude, y_test_ballistic_longitude = train_test_split(X_Ballistic, y_Full_Ballistic_longitude, test_size=0.20, random_state=42)

# Create the empty brains
# MainAtApogee
model_main_distance, model_main_latitude, model_main_longitude = XGBRegressor(), XGBRegressor(), XGBRegressor()

# DrogueOnly
model_drogue_distance, model_drogue_latitude, model_drogue_longitude = XGBRegressor(), XGBRegressor(), XGBRegressor()

# Ballistic
model_ballistic_distance, model_ballistic_latitude, model_ballistic_longitude = XGBRegressor(), XGBRegressor(), XGBRegressor()

In [32]:
print([col for col in X_train_main.columns if X_train_main.columns.tolist().count(col) > 1])


[]


In [33]:
# The training 

# MainAtApogee
model_main_distance.fit(X_train_main, y_train_main_distance)
model_main_latitude.fit(X_train_main, y_train_main_latitude)
model_main_longitude.fit(X_train_main, y_train_main_longitude)

# DrogueOnly
model_drogue_distance.fit(X_train_drogue, y_train_drogue_distance)
model_drogue_latitude.fit(X_train_drogue, y_train_drogue_latitude)
model_drogue_longitude.fit(X_train_drogue, y_train_drogue_longitude)

# Ballistic
model_ballistic_distance.fit(X_train_ballistic, y_train_ballistic_distance)
model_ballistic_latitude.fit(X_train_ballistic, y_train_ballistic_latitude)
model_ballistic_longitude.fit(X_train_ballistic, y_train_ballistic_longitude)

,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None
,feature_types,None
,gamma,None


In [34]:
# MainAtApogee
y_pred_main_distance = model_main_distance.predict(X_test_main)
y_pred_main_latitude = model_main_latitude.predict(X_test_main)
y_pred_main_longitude = model_main_longitude.predict(X_test_main)

# DrogueOnly
y_pred_drogue_distance = model_drogue_distance.predict(X_test_drogue)
y_pred_drogue_latitude = model_drogue_latitude.predict(X_test_drogue)
y_pred_drogue_longitude = model_drogue_longitude.predict(X_test_drogue)

# Ballistic
y_pred_ballistic_distance = model_ballistic_distance.predict(X_test_ballistic)
y_pred_ballistic_latitude = model_ballistic_latitude.predict(X_test_ballistic)
y_pred_ballistic_longitude = model_ballistic_longitude.predict(X_test_ballistic)

In [35]:
from sklearn.metrics import mean_squared_error, r2_score

"""
Root mean squared error: how off the models' predictions were objectively from the real result
R2 (squared) score: the general percentage of accuracy to the real results
"""

# MainAtApogee
model_main_distance_rms = np.sqrt(mean_squared_error(y_test_main_distance, y_pred_main_distance))
model_main_latitude_rms = np.sqrt(mean_squared_error(y_test_main_latitude, y_pred_main_latitude))
model_main_longitude_rms = np.sqrt(mean_squared_error(y_test_main_longitude, y_pred_main_longitude))
model_main_distance_r2 = r2_score(y_test_main_distance, y_pred_main_distance)
model_main_latitude_r2 = r2_score(y_test_main_latitude, y_pred_main_latitude)
model_main_longitude_r2 = r2_score(y_test_main_longitude, y_pred_main_longitude)

print(f"MainAtApogee | Distance RMS: {model_main_distance_rms} | R2: {model_main_distance_r2}")
print(f"MainAtApogee | Latitude RMS: {model_main_latitude_rms} | R2: {model_main_latitude_r2}")
print(f"MainAtApogee | Longitude RMS: {model_main_longitude_rms} | R2: {model_main_longitude_r2}")

# DrogueOnly
model_drogue_distance_rms = np.sqrt(mean_squared_error(y_test_drogue_distance, y_pred_drogue_distance))
model_drogue_latitude_rms = np.sqrt(mean_squared_error(y_test_drogue_latitude, y_pred_drogue_latitude))
model_drogue_longitude_rms = np.sqrt(mean_squared_error(y_test_drogue_longitude, y_pred_drogue_longitude))
model_drogue_distance_r2 = r2_score(y_test_drogue_distance, y_pred_drogue_distance)
model_drogue_latitude_r2 = r2_score(y_test_drogue_latitude, y_pred_drogue_latitude)
model_drogue_longitude_r2 = r2_score(y_test_drogue_longitude, y_pred_drogue_longitude)

print(f"DrogueOnly   | Distance RMS: {model_drogue_distance_rms} | R2: {model_drogue_distance_r2}")
print(f"DrogueOnly   | Latitude RMS: {model_drogue_latitude_rms} | R2: {model_drogue_latitude_r2}")
print(f"DrogueOnly   | Longitude RMS: {model_drogue_longitude_rms} | R2: {model_drogue_longitude_r2}")

# Ballistic
model_ballistic_distance_rms = np.sqrt(mean_squared_error(y_test_ballistic_distance, y_pred_ballistic_distance))
model_ballistic_latitude_rms = np.sqrt(mean_squared_error(y_test_ballistic_latitude, y_pred_ballistic_latitude))
model_ballistic_longitude_rms = np.sqrt(mean_squared_error(y_test_ballistic_longitude, y_pred_ballistic_longitude))
model_ballistic_distance_r2 = r2_score(y_test_ballistic_distance, y_pred_ballistic_distance)
model_ballistic_latitude_r2 = r2_score(y_test_ballistic_latitude, y_pred_ballistic_latitude)
model_ballistic_longitude_r2 = r2_score(y_test_ballistic_longitude, y_pred_ballistic_longitude)

print(f"Ballistic    | Distance RMS: {model_ballistic_distance_rms} | R2: {model_ballistic_distance_r2}")
print(f"Ballistic    | Latitude RMS: {model_ballistic_latitude_rms} | R2: {model_ballistic_latitude_r2}")
print(f"Ballistic    | Longitude RMS: {model_ballistic_longitude_rms} | R2: {model_ballistic_longitude_r2}")

MainAtApogee | Distance RMS: 0.5858301114856143 | R2: 0.8996497624890581
MainAtApogee | Latitude RMS: 0.013680966906897536 | R2: 0.9621876071359217
MainAtApogee | Longitude RMS: 0.012428701082011785 | R2: 0.9712506706084887
DrogueOnly   | Distance RMS: 0.23628230286700402 | R2: 0.9502389404281419
DrogueOnly   | Latitude RMS: 0.005716141721703922 | R2: 0.9047434810632822
DrogueOnly   | Longitude RMS: 0.009675653236572504 | R2: 0.9121390541940287
Ballistic    | Distance RMS: 0.22942026290628612 | R2: 0.9758623847781123
Ballistic    | Latitude RMS: 0.009358922728676347 | R2: 0.8454496048273188
Ballistic    | Longitude RMS: 0.007594676340882421 | R2: 0.9609267032219528


In [36]:
import joblib
import os

absolute_path = "/Users/ricechessmaster/Documents/Dispersion Analysis v2/backend/ml_models"

models = {
    "model_main_distance": model_main_distance,
    "model_main_latitude": model_main_latitude,
    "model_main_longitude": model_main_longitude,
    "model_drogue_distance": model_drogue_distance,
    "model_drogue_latitude": model_drogue_latitude,
    "model_drogue_longitude": model_drogue_longitude,
    "model_ballistic_distance": model_ballistic_distance,
    "model_ballistic_latitude": model_ballistic_latitude,
    "model_ballistic_longitude": model_ballistic_longitude,
}

for name, model in models.items():
    full_path = os.path.join(absolute_path, f"{name}.joblib")
    joblib.dump(model, full_path)
    print(f"Saved {name} -> {full_path}")

Saved model_main_distance -> /Users/ricechessmaster/Documents/Dispersion Analysis v2/backend/ml_models/model_main_distance.joblib
Saved model_main_latitude -> /Users/ricechessmaster/Documents/Dispersion Analysis v2/backend/ml_models/model_main_latitude.joblib
Saved model_main_longitude -> /Users/ricechessmaster/Documents/Dispersion Analysis v2/backend/ml_models/model_main_longitude.joblib
Saved model_drogue_distance -> /Users/ricechessmaster/Documents/Dispersion Analysis v2/backend/ml_models/model_drogue_distance.joblib
Saved model_drogue_latitude -> /Users/ricechessmaster/Documents/Dispersion Analysis v2/backend/ml_models/model_drogue_latitude.joblib
Saved model_drogue_longitude -> /Users/ricechessmaster/Documents/Dispersion Analysis v2/backend/ml_models/model_drogue_longitude.joblib
Saved model_ballistic_distance -> /Users/ricechessmaster/Documents/Dispersion Analysis v2/backend/ml_models/model_ballistic_distance.joblib
Saved model_ballistic_latitude -> /Users/ricechessmaster/Documen